У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

# Task 1
**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [519]:
# imports
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC, SMOTEN
from imblearn.combine import SMOTETomek
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

In [520]:
df_raw = pd.read_csv('customer_segmentation_train.csv')

In [521]:
df_raw.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [522]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [523]:
input_df = df_raw[df_raw.columns[1:-1]].copy()
input_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
0,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4
1,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4
2,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6
3,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6
4,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6


In [524]:
target_df = df_raw[df_raw.columns[-1]].copy()

# Imputation of missing values

## `Ever_Married`:

In [525]:
input_df['Ever_Married'].value_counts()

,count
Ever_Married,
Yes,4643
No,3285


In [526]:
# percentage of missing values
np.round(((input_df['Ever_Married'].isna().sum()/input_df.shape[0]) * 100), 2)

np.float64(1.74)

In [527]:
# indicator column to mark imputed observations
input_df['Ever_Married_Imputed'] = input_df['Ever_Married'].isna().astype(int)

In [528]:
# fill in misisng values with mode based on age
input_df['Ever_Married'] = input_df.groupby('Age')['Ever_Married'].transform(lambda x: x.fillna(x.mode()[0]))

In [529]:
input_df['Ever_Married'].value_counts()

,count
Ever_Married,
Yes,4725
No,3343


## `Graduated`:

In [530]:
input_df['Graduated'].value_counts()

,count
Graduated,
Yes,4968
No,3022


In [531]:
# percentage of missing values
np.round((input_df['Graduated'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(0.97)

In [532]:
# fill misisng values with 'Unknown'
input_df['Graduated'] = input_df['Graduated'].fillna(value='Unknown')

In [533]:
input_df['Graduated'].value_counts()

,count
Graduated,
Yes,4968
No,3022
Unknown,78


## `Profession`:

In [534]:
input_df['Profession'].value_counts()

,count
Profession,
Artist,2516
Healthcare,1332
Entertainment,949
Engineer,699
Doctor,688
Lawyer,623
Executive,599
Marketing,292
Homemaker,246


In [535]:
# percentage of missing values
np.round((input_df['Profession'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(1.54)

In [536]:
# fill misisng values with 'Unknown'
input_df.fillna({'Profession': 'Unknown'}, inplace=True)

In [537]:
input_df.Profession.value_counts()

,count
Profession,
Artist,2516
Healthcare,1332
Entertainment,949
Engineer,699
Doctor,688
Lawyer,623
Executive,599
Marketing,292
Homemaker,246


## `Work_Experience`:

In [538]:
input_df.Work_Experience.value_counts()

,count
Work_Experience,
1.0,2354
0.0,2318
9.0,474
8.0,463
2.0,286
3.0,255
4.0,253
6.0,204
7.0,196


In [539]:
# percentage of missing values
np.round((input_df['Work_Experience'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(10.28)

In [540]:
# indicator column to mark imputed observations
input_df['Work_Experience_Imputed'] = input_df['Work_Experience'].isna().astype(int)

In [541]:
# impute missing values with median work experience of corresponding age
input_df['Work_Experience'] = input_df.groupby('Age')['Work_Experience'].transform(lambda x: x.fillna(x.median()))

In [542]:
# check percentage of missing values again - expected to be 0%
np.round((input_df['Work_Experience'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(0.0)

## `Family_Size`:

In [543]:
input_df['Family_Size'].value_counts()

,count
Family_Size,
2.0,2390
3.0,1497
1.0,1453
4.0,1379
5.0,612
6.0,212
7.0,96
8.0,50
9.0,44


In [544]:
# percentage of missing values
np.round((input_df['Family_Size'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(4.15)

In [545]:
# indicator column to mark imputed observations
input_df['Family_Size_Imputed'] = input_df['Family_Size'].isna().astype(int)

In [546]:
# impute missing values with median family size of corresponding age
input_df['Family_Size'] = input_df.groupby('Age')['Family_Size'].transform(lambda x: x.fillna(x.median()))

In [547]:
# check percentage of missing values again - expected to be 0%
np.round((input_df['Family_Size'].isna().sum()/input_df.shape[0])*100)

np.float64(0.0)

## `Var_1`:

In [548]:
input_df['Var_1'].value_counts()

,count
Var_1,
Cat_6,5238
Cat_4,1089
Cat_3,822
Cat_2,422
Cat_7,203
Cat_1,133
Cat_5,85


In [549]:
# percentage of missing values
np.round((input_df['Var_1'].isna().sum()/input_df.shape[0])*100, 2)

np.float64(0.94)

In [550]:
# indicator column to mark imputed observations
input_df['Var_1_Imputed'] = input_df['Var_1'].isna().astype(int)

In [551]:
# fill misisng values with "Unknown"
input_df['Var_1'] = input_df['Var_1'].fillna(value='Unknown')

In [552]:
# check once again the values
input_df['Var_1'].value_counts()

,count
Var_1,
Cat_6,5238
Cat_4,1089
Cat_3,822
Cat_2,422
Cat_7,203
Cat_1,133
Cat_5,85
Unknown,76


In [553]:
# to check if all missing values are imputed
input_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   8068 non-null   object 
 1   Ever_Married             8068 non-null   object 
 2   Age                      8068 non-null   int64  
 3   Graduated                8068 non-null   object 
 4   Profession               8068 non-null   object 
 5   Work_Experience          8068 non-null   float64
 6   Spending_Score           8068 non-null   object 
 7   Family_Size              8068 non-null   float64
 8   Var_1                    8068 non-null   object 
 9   Ever_Married_Imputed     8068 non-null   int64  
 10  Work_Experience_Imputed  8068 non-null   int64  
 11  Family_Size_Imputed      8068 non-null   int64  
 12  Var_1_Imputed            8068 non-null   int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 819.5+ KB


# Encoding categorical data

In [554]:
input_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed
0,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,0,0,0,0
1,Female,Yes,38,Yes,Engineer,1.0,Average,3.0,Cat_4,0,1,0,0
2,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,0,0,0,0
3,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,0,0,0,0
4,Female,Yes,40,Yes,Entertainment,1.0,High,6.0,Cat_6,0,1,0,0


In [555]:
# mapping binary column Ever_Maried to ensure No->0, Yes->1
input_df['Ever_Married'] = input_df['Ever_Married'].map({'No':0, 'Yes':1})

In [556]:
# setup one-hot encoder for categorical data
one_hot_enc = OneHotEncoder(handle_unknown='ignore', drop='first')
one_hot_enc.fit(input_df[['Gender', 'Graduated', 'Profession', 'Var_1']])

OneHotEncoder(drop='first', handle_unknown='ignore')

In [557]:
one_hot_enc.categories_

[array(['Female', 'Male'], dtype=object),
 array(['No', 'Unknown', 'Yes'], dtype=object),
 array(['Artist', 'Doctor', 'Engineer', 'Entertainment', 'Executive',
        'Healthcare', 'Homemaker', 'Lawyer', 'Marketing', 'Unknown'],
       dtype=object),
 array(['Cat_1', 'Cat_2', 'Cat_3', 'Cat_4', 'Cat_5', 'Cat_6', 'Cat_7',
        'Unknown'], dtype=object)]

In [558]:
# perform one-hot encoding
one_hot = one_hot_enc.transform(input_df[['Gender', 'Graduated', 'Profession', 'Var_1']]).toarray()
one_hot

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       ...,
       [0., 0., 1., ..., 1., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.]])

In [559]:
cols = one_hot_enc.get_feature_names_out(['Gender', 'Graduated', 'Profession', 'Var_1'])

In [560]:
# add encoded data into dataframe
input_df[cols] = one_hot

In [561]:
pd.set_option('display.max_columns', None)
input_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown
0,Male,0,22,No,Healthcare,1.0,Low,4.0,Cat_4,0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,Female,1,38,Yes,Engineer,1.0,Average,3.0,Cat_4,0,1,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,Female,1,67,Yes,Engineer,1.0,Low,1.0,Cat_6,0,0,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,Male,1,67,Yes,Lawyer,0.0,High,2.0,Cat_6,0,0,0,0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,Female,1,40,Yes,Entertainment,1.0,High,6.0,Cat_6,0,1,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [562]:
# setup ordinal encoder for ordinal data
ord_enc = OrdinalEncoder(categories=[['Low', 'Average', 'High']])
ord_enc.fit(input_df[['Spending_Score']])

OrdinalEncoder(categories=[['Low', 'Average', 'High']])

In [563]:
# perform ordinal encoding
ord_data = ord_enc.transform(input_df[['Spending_Score']])

In [564]:
# add encoded data into dataframe
input_df['Spending_Score_Encoded'] = ord_data

In [565]:
input_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown,Spending_Score_Encoded
0,Male,0,22,No,Healthcare,1.0,Low,4.0,Cat_4,0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,Female,1,38,Yes,Engineer,1.0,Average,3.0,Cat_4,0,1,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,Female,1,67,Yes,Engineer,1.0,Low,1.0,Cat_6,0,0,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,Male,1,67,Yes,Lawyer,0.0,High,2.0,Cat_6,0,0,0,0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
4,Female,1,40,Yes,Entertainment,1.0,High,6.0,Cat_6,0,1,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0


In [566]:
# select numeric and categorical columns (names)
numeric_cols = input_df.select_dtypes('number').columns.tolist()
categorical_cols = input_df.select_dtypes('object').columns.tolist()

print(f'Numeric col. names: {numeric_cols}')
print(f'Categorical col. name: {categorical_cols}')

Numeric col. names: ['Ever_Married', 'Age', 'Work_Experience', 'Family_Size', 'Ever_Married_Imputed', 'Work_Experience_Imputed', 'Family_Size_Imputed', 'Var_1_Imputed', 'Gender_Male', 'Graduated_Unknown', 'Graduated_Yes', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Profession_Unknown', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7', 'Var_1_Unknown', 'Spending_Score_Encoded']
Categorical col. name: ['Gender', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']


In [567]:
# define input columns for the model
target_col = df_raw.columns.tolist()[-1]
target_col

'Segmentation'

# Scaling numeric data

In [568]:
# setup min-max scaler for numeric columns
minmax_scaler = MinMaxScaler()
minmax_scaler.fit(input_df[numeric_cols])

MinMaxScaler()

In [569]:
# scale data
input_df[numeric_cols] = minmax_scaler.transform(input_df[numeric_cols])

In [570]:
input_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown,Spending_Score_Encoded
0,Male,0.0,0.056338,No,Healthcare,0.071429,Low,0.375,Cat_4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,Female,1.0,0.281690,Yes,Engineer,0.071429,Average,0.250,Cat_4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.5
2,Female,1.0,0.690141,Yes,Engineer,0.071429,Low,0.000,Cat_6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,Male,1.0,0.690141,Yes,Lawyer,0.000000,High,0.125,Cat_6,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,Female,1.0,0.309859,Yes,Entertainment,0.071429,High,0.625,Cat_6,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


# Splitting data into train and test

In [571]:
input_df.insert(len(input_df.columns), 'Segmentation', target_df)

In [572]:
train_df, test_df = train_test_split(input_df, test_size=0.2, random_state=42, stratify=input_df.Segmentation)

In [573]:
train_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown,Spending_Score_Encoded,Segmentation
917,Female,0.0,0.197183,Yes,Artist,0.642857,Low,0.000,Cat_6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,A
3398,Male,1.0,0.760563,Yes,Entertainment,0.071429,Average,0.125,Cat_6,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5,B
2045,Female,0.0,0.211268,Yes,Entertainment,0.071429,Low,0.375,Cat_6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,B
8060,Female,1.0,0.422535,Yes,Artist,0.000000,Average,0.625,Cat_6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5,A
4604,Female,1.0,0.140845,No,Doctor,0.642857,Low,0.000,Cat_7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,A


In [575]:
# train and test sets containing only numeric data
X_train = train_df[numeric_cols]
X_test = test_df[numeric_cols]

In [576]:
X_train.head()

,Ever_Married,Age,Work_Experience,Family_Size,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown,Spending_Score_Encoded
917,0.0,0.197183,0.642857,0.000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3398,1.0,0.760563,0.071429,0.125,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
2045,0.0,0.211268,0.071429,0.375,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8060,1.0,0.422535,0.000000,0.625,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
4604,1.0,0.140845,0.642857,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [577]:
y_train = train_df[target_col]
y_test = test_df[target_col]

# Task 2

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [581]:
# check the traget classes in training set
y_train.value_counts()

,count
Segmentation,
D,1814
A,1578
C,1576
B,1486


## SMOTENC

In [582]:
# check data types -> decide which to treat as categorical (where interpolated values should stay natural numbers)
X_train.head()

,Ever_Married,Age,Work_Experience,Family_Size,Ever_Married_Imputed,Work_Experience_Imputed,Family_Size_Imputed,Var_1_Imputed,Gender_Male,Graduated_Unknown,Graduated_Yes,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Profession_Unknown,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Var_1_Unknown,Spending_Score_Encoded
917,0.0,0.197183,0.642857,0.000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3398,1.0,0.760563,0.071429,0.125,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
2045,0.0,0.211268,0.071429,0.375,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8060,1.0,0.422535,0.000000,0.625,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
4604,1.0,0.140845,0.642857,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


У цьому наборі даних, після кодування категоріальних колонок, містяться колонки з наступними типами даних:
- номінальні (цілочисельні/натуральні): `Age`, `Work_Experience`, `Family_Size`
- ординальні: `Spending_Score_Encoded`
- бінарні: решта колонок.

На мою думку, всі ці колонки треба обробляти як категоріальні при виконанні ресемплінгу (метод SMOTEN), щоб не допустити розбиття їх значень на дроби -- щоб значення після ресемплінгу відповідали (немасштабованим) значенням/рівням даних ознак. Але у даному випадку, щоб мати змогу застосувати метод SMOTENC який вимагає як мінімум одну колонку ознак з числовим типом даних (дійсні значення), ознаку `Age` вважатимемо такою, що може приймати числові дійсні значення.

In [583]:
categorical_cols_smotenc = X_train.columns.tolist()
categorical_cols_smotenc.remove('Age')

In [584]:
categorical_cols_smotenc

['Ever_Married',
 'Work_Experience',
 'Family_Size',
 'Ever_Married_Imputed',
 'Work_Experience_Imputed',
 'Family_Size_Imputed',
 'Var_1_Imputed',
 'Gender_Male',
 'Graduated_Unknown',
 'Graduated_Yes',
 'Profession_Doctor',
 'Profession_Engineer',
 'Profession_Entertainment',
 'Profession_Executive',
 'Profession_Healthcare',
 'Profession_Homemaker',
 'Profession_Lawyer',
 'Profession_Marketing',
 'Profession_Unknown',
 'Var_1_Cat_2',
 'Var_1_Cat_3',
 'Var_1_Cat_4',
 'Var_1_Cat_5',
 'Var_1_Cat_6',
 'Var_1_Cat_7',
 'Var_1_Unknown',
 'Spending_Score_Encoded']

In [585]:
# get indices of categorical columns for SMOTENC
cat_cols_indices = X_train.columns.get_indexer(categorical_cols_smotenc)
cat_cols_indices

array([ 0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27])

In [586]:
# setup SMOTENC
smotenc = SMOTENC(random_state=42, categorical_features=list(cat_cols_indices))

In [587]:
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)

In [588]:
train_df.shape, X_train_smotenc.shape, y_train_smotenc.shape

((6454, 34), (7256, 28), (7256,))

In [589]:
# check the amounts of exemplars of each target class after resampling
y_train_smotenc.value_counts()

,count
Segmentation,
A,1814
B,1814
C,1814
D,1814


Після ресемплінгу у ознаки `Age` збільшилась кількість рівнів: з 67 до 752 відповідно.

In [590]:
X_train.Age.value_counts()

,count
Age,
0.239437,199
0.267606,193
0.309859,189
0.352113,189
0.281690,185
...,...
0.816901,22
0.957746,20
0.830986,20


In [591]:
X_train_smotenc.Age.value_counts()

,count
Age,
0.239437,206
0.267606,197
0.309859,195
0.352113,193
0.281690,188
...,...
0.528083,1
0.544723,1
0.435220,1


## SMOTE-Tomek (SMOTEK)

In [592]:
smotetomek = SMOTETomek(random_state=42)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train, y_train)

In [593]:
# check the amounts of exemplars of each target class after resampling
y_train_smotetomek.value_counts()

,count
Segmentation,
C,1454
D,1453
B,1411
A,1384


# Task 3

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

## Original data

In [594]:
# Logistic regression setup: one-vs-rest (OvR)
log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train, y_train)
ovr_predictions = ovr_model.predict(X_test)

In [595]:
# calculate precision & recall for each class
print(classification_report(y_test, ovr_predictions))

              precision    recall  f1-score   support

           A       0.41      0.45      0.43       394
           B       0.37      0.14      0.20       372
           C       0.48      0.63      0.54       394
           D       0.64      0.75      0.69       454

    accuracy                           0.51      1614
   macro avg       0.48      0.49      0.47      1614
weighted avg       0.48      0.51      0.48      1614



## SMOTENC data

In [596]:
# Logistic regression setup: one-vs-rest (OvR)
log_reg_smotenc = LogisticRegression(solver='liblinear')
ovr_model_smotenc = OneVsRestClassifier(log_reg_smotenc)
ovr_model_smotenc.fit(X_train_smotenc, y_train_smotenc)
ovr_predictions_smotenc = ovr_model_smotenc.predict(X_test)

In [597]:
# calculate precision & recall for each class
print(classification_report(y_test, ovr_predictions_smotenc))

              precision    recall  f1-score   support

           A       0.42      0.48      0.45       394
           B       0.40      0.20      0.27       372
           C       0.50      0.61      0.55       394
           D       0.66      0.71      0.69       454

    accuracy                           0.51      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614



## SMOTETomek data

In [598]:
# Logistic regression setup: one-vs-rest (OvR)
log_reg_smotetomek = LogisticRegression(solver='liblinear')
ovr_model_smotetomek = OneVsRestClassifier(log_reg_smotetomek)
ovr_model_smotetomek.fit(X_train_smotetomek, y_train_smotetomek)
ovr_predictions_smotetomek = ovr_model_smotetomek.predict(X_test)

In [599]:
# calculate precision & recall for each class
print(classification_report(y_test, ovr_predictions_smotetomek))

              precision    recall  f1-score   support

           A       0.41      0.47      0.44       394
           B       0.38      0.21      0.27       372
           C       0.49      0.60      0.54       394
           D       0.68      0.70      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614



In [600]:
y_train.value_counts()

,count
Segmentation,
D,1814
A,1578
C,1576
B,1486


In [601]:
y_train_smotenc.value_counts()

,count
Segmentation,
A,1814
B,1814
C,1814
D,1814


In [602]:
y_train_smotetomek.value_counts()

,count
Segmentation,
C,1454
D,1453
B,1411
A,1384


**Висновок:**

Для порівняння моделей використаємо метрику F-1.
Якщо порівнювати метрику по кожному з класів, то ми бачимо, що клас В класифікується найгірше у всіх трьох випадках - це потребує окремого дослідження причин.
Загалом, усі три моделі показали порівняно однаковий не дуже гарний результат - середнє занчення F-1 по всіх класах: 0.47 (без ресемплінгу), 0.49 (SMOTENC), 0.48 (SMOTETomek). На мою думку, найбільш вагомою причиною відсутності суттєвої різниці у якості передбачення моделей є те, що дисбаланс класів у початковому наборі даних є не суттєвим і, відповідно, ресемплінг з метою покращення балансу класів не приніс достатньої користі для якості передбачення.
